In [31]:

# load the liabries
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import meteostat as ms
import holidays

from loader import LoaderStorage
from constants import AIRPORT_LIMIT_LIST

storage = LoaderStorage(
  root="s3://data-mining/"
)

# the airlines are already filterd to the ones only that we use.
source = "data/interim/" # set folders for laoding the Data from
destination = "data/features/" # where to save the data with new Features
output_file_name = "feature_engineered.parquet"

In [32]:
Dataset_name = "1_year_data_new.parquet"
# load the dataset
dataset = storage.read_parquet(source + Dataset_name)
num_rows = len(dataset)
# iterate through all the columsn and if there are any nan values, we print the column name and the number of nan values

# cut of year, from where we will start using the data for training, before that is just used for feature engneering.
min_year = 2014
hours_before = 2 #the information time is 2h before the departure time, so we need to get the last hour mark before that time to merge with the weather data, as the weather data is only available on an hourly basis.
dataset["FlightID"] = dataset.index
dataset['DayOfYear'] = dataset['CRSDepDateTime'].dt.dayofyear

display(dataset.head())

,Unnamed: 0,Year,Month,DayofMonth,DayOfWeek,FlightDate,Reporting_Airline,Tail_Number,Flight_Number_Reporting_Airline,Origin,...,DepDateTime,DepDateTime_UTC,ArrDateTime,ArrDateTime_UTC,Information_time_UTC,floor_informationtime_UTC,Information_time,floor_informationtime,FlightID,DayOfYear
0,0,2014,1,30,4,2014-01-30,AA,N006AA,2377,DFW,...,2014-01-30 09:35:00,2014-01-30 15:35:00+00:00,2014-01-30 10:51:00,2014-01-30 16:51:00+00:00,2014-01-30 13:40:00,2014-01-30 13:00:00,2014-01-30 07:40:00,2014-01-30 07:00:00,0,30
1,1,2014,1,31,5,2014-01-31,AA,N003AA,2377,DFW,...,2014-01-31 09:51:00,2014-01-31 15:51:00+00:00,2014-01-31 11:15:00,2014-01-31 17:15:00+00:00,2014-01-31 13:40:00,2014-01-31 13:00:00,2014-01-31 07:40:00,2014-01-31 07:00:00,1,31
2,2,2014,1,1,3,2014-01-01,AA,N002AA,2377,ICT,...,2014-01-01 11:44:00,2014-01-01 17:44:00+00:00,2014-01-01 13:02:00,2014-01-01 19:02:00+00:00,2014-01-01 15:35:00,2014-01-01 15:00:00,2014-01-01 09:35:00,2014-01-01 09:00:00,2,1
3,3,2014,1,2,4,2014-01-02,AA,N002AA,2377,ICT,...,2014-01-02 11:34:00,2014-01-02 17:34:00+00:00,2014-01-02 12:53:00,2014-01-02 18:53:00+00:00,2014-01-02 15:35:00,2014-01-02 15:00:00,2014-01-02 09:35:00,2014-01-02 09:00:00,3,2
4,4,2014,1,3,5,2014-01-03,AA,N014AA,2377,ICT,...,2014-01-03 11:29:00,2014-01-03 17:29:00+00:00,2014-01-03 12:44:00,2014-01-03 18:44:00+00:00,2014-01-03 15:35:00,2014-01-03 15:00:00,2014-01-03 09:35:00,2014-01-03 09:00:00,4,3


# Feature Engineering

### 1. Real time avg delay infos
this Cell tracks real time avergae delays, and number of flights, developing over the day, and feeding the live number in the Information time (2h pre CRSDep), to the flight information.
There are a couple steps to this, and nuances:
1. We sort by day, and add the infromation to every flight, how manny flights have deported so far, and some more information.
2. we Group the this information by the Hour to have this information just refreshed for every Hour.
3. And We merge the calulated "Day so far" Values, on the last full hour before the information time.


### 1.1 Orgin Delay Live Departure features

> these are features, about the avgerge departure dealys, number of departed flights, at the origin airport. 

- for example the avergae delay, of the whole day so far (sfd), untill the last full hours before 2h before sheduled takeoff.
- we get information about all aircraft which have departed before that from the same airport

In [ ]:
# a smaller version, so we only need to merge the rolling avg info to col, we will keep later.
dataset_airport_limit = dataset[
    dataset['Origin'].isin(AIRPORT_LIMIT_LIST)
    & dataset['Dest'].isin(AIRPORT_LIMIT_LIST)
    & (dataset['Cancelled'] == 0)
    & (dataset['Year'] >= min_year)
].copy()

# new featureengeniered delay information: delay that happend while fliying, (might have to do with dest airport)
dataset['route_delay'] = dataset['ArrDelayMinutes'] - dataset['DepDelayMinutes']

def rolliing_day_so_far_data(dataset, airport_grouping, Day_column, time_column, Delay_collumns, prefix):
    """
    dataset: origianl dataset
    airport grouping: "Dest" or "Orgin", depending on the data we collect
    Day_column: the column that contains the day information, in our case "FlightDate" or a UTC_corrected version, if local time doesnt work.
    time_column: the column that contains the time information, in our case "DepDateTime" or "ArrDateTime", or UTC_corrected versions, if local time doesnt work.
    Delay_collumns: the collumns that contain the delay information, in our case "DepDelayMinutes" or "ArrDelayMinutes", or further delay related collumns
    prefix: the prefix for the features
    """

    # code is hard to read because of speed optimization
    required_cols = [airport_grouping, Day_column, time_column] + Delay_collumns + ['Cancelled', 'Year']
    # Source rows used to compute historical delay at each airport/day, limited in collumns to speed up the process
    src = dataset[required_cols].copy()
    # drop all rows where the flight was in a year to early for our dataset, or where the flight was cancelled, as we can not use these flights to calculate the historical delay at each airport/day
    src = src[(src['Year'] >= min_year)]
    #also drop canncelled flights since they might cause issuees
    src = src[src['Cancelled'] == 0]

    #variable to save the names of the new collums.
    col_names =[]

    # fill the nan of the delay columns
    for coll in Delay_collumns:
        src[coll] = src[coll].fillna(0)

    # sor the dataset by Origin, and the Local DateTime of ACTUAL Departure
    src = src.sort_values([airport_grouping, Day_column, time_column])
    # group the Values Per Origin Airport and Day, to create cumulative values for each group
    grp = src.groupby([airport_grouping, Day_column], sort=False) 

    # -> Now we willl create cumulative values for each group to get the historical delay information.
    # the inforamtion is the current total that day until that moment.
    src['tmp_did_depart'] = src[time_column].notna().astype(int) # if there is an actual departure time, the flight did depart, otherwise it did not depart
    src[f'{prefix}_cum_count_sfd'] = grp['tmp_did_depart'].cumsum() # the number of flights before this flight (including this flight), in the group
    col_names.extend([f'{prefix}_cum_count_sfd'])
    # the cumlative delays
    for coll in Delay_collumns:
        src[f'{prefix}_cum_{coll}_sfd'] = grp[coll].cumsum() # the cumulative delay of all flights before, including this flight (in the Group)
        # calculate the average delay for each airport/day, by dividing the cumulative delay by the cumulative count of flights
        src[f'{prefix}_avg_{coll}_sfd'] = src[f'{prefix}_cum_{coll}_sfd'] / src[f'{prefix}_cum_count_sfd'] # the average delay of all flights before, including this flight (in the Group)
        src[f'{prefix}_avg_{coll}_sfd'] = src[f'{prefix}_avg_{coll}_sfd'].fillna(0) # if there are no flights, we set the average delay to 0, to avoid division by zero
        col_names.extend([f'{prefix}_cum_{coll}_sfd', f'{prefix}_avg_{coll}_sfd']) 


    # group the information by Origin, Flight date, and the next hour (ceil hour) of the departure time, to get the cumulative delay and count of flights for each airport, here always use the row where the time is the latest or the count is highest.
    src['time_ceil'] = src[time_column].dt.ceil('h') # get the next hour for every flight.
    # Only kee the last flight of the Hour, which has the final cumsum for the hour window.
    src = src.sort_values([airport_grouping, Day_column, 'time_ceil', time_column], ascending=[True, True, True, False]) # for order to have for every same Ceil hour the flight with the highest minutes number (last flight in the hour) first
    # get some information about the average delays, just in the last hour before the information hour.
    hour_group = src.groupby([airport_grouping, Day_column, 'time_ceil'], sort=False)
    
    # specific one hour averages and number of flights count. -> only for one hour window.
    src[f'{prefix}_count_fligts_lh'] = hour_group['tmp_did_depart'].sum().reset_index(drop=True)
       
    col_names.extend([f'{prefix}_count_fligts_lh'])
    # remove the grouping from the serires we just injected 
    for coll in Delay_collumns:
        src[f'{prefix}_avg_{coll}_lh'] = hour_group[coll].mean().reset_index(drop=True)
        col_names.extend([f'{prefix}_avg_{coll}_lh'])

    # back to the day rolling averges, we only keep the last flight of the hour, which has the final cumsum and averages, of the hour window.
    src = src.drop_duplicates(subset=[airport_grouping, Day_column, 'time_ceil'], keep='first') # only keep the last flight in the hour

    # group by also the hour now, and get the las row for the rolling day averges so far
    # but also get some 
    grp = src.groupby([airport_grouping, Day_column, 'time_ceil'], sort=False)
    # for every hour i want the last information about the cumulative delay and count of flights, so i want to get the last row for each group
    src = grp.last().reset_index()
    # also get the number of flights that departed in the last hour, to get the cumulative count of flights for each airport, here always use the row where the time is the latest or the count is highest.
    src['flight_count_sfd'] = grp['tmp_did_depart'].sum().reset_index(drop=True)

    #the columns to drop now:
    drop_cols = ['tmp_did_depart']
    src = src.drop(columns=drop_cols)


    # to fill missing hours, create a dataframe with all hours, days and all airports
    min_date = src['time_ceil'].min()
    max_date = src['time_ceil'].max()
    all_hours = pd.date_range(start=min_date, end=max_date, freq='h')
    all_airports = src[airport_grouping].unique()
    all_airport_hours = pd.MultiIndex.from_product([all_airports, all_hours], names=[airport_grouping, 'time_ceil']).to_frame(index=False)
    
    # merge with the original dataframe to get all the hours for each airport
    src = all_airport_hours.merge(src, on=[airport_grouping, 'time_ceil'], how='left')
    # for teh cumaltive day so far colums, we need to fill in the hours where there where no flights, since these do not have a datapoint now.
    # we fill these value by forward filling, untill we would hit a day barrier, then we fill with 0
    src = src.sort_values([airport_grouping, Day_column, 'time_ceil'])
    day_group = src.groupby([airport_grouping, Day_column], sort=False)
    # avergae and cum cols
    sfd_cols = [col for col in col_names if 'sfd' in col]
    src[sfd_cols] = day_group[sfd_cols].ffill().fillna(0) #for the cumulatives
    
    # the columsn we still need of src:
    # src = src[['Origin', 'FlightDate', 'DepDateTime_ceil', 'avg_delay', 'cum_count','cum_cannceld_flights', 'flights_so_far_day']]

    # rename the columsn we mereg on to aviod column name conflicts.
    # rename the airport grouping and day column to merge with the original dataset later.
    src = src.rename(columns={airport_grouping: f"{airport_grouping}_src", Day_column: f"{Day_column}_src"})
    print(col_names)
    return src,col_names

# only get this information for the rows we actually need


depature_rolling_averages_cols =["DepDelayMinutes"]
src,col_names = rolliing_day_so_far_data(dataset, "Origin", "FlightDate", "DepDateTime", depature_rolling_averages_cols, "R_dep")
display(src.head())
# we merge the information of src with the original dataset, by merging on the Origin, FlightDate and the next hour (ceil hour) of the departure time, to get the cumulative delay and count of flights for each airport, here always use the row where the time is the latest or the count is highest.
df_rollling_avg = dataset_airport_limit.merge(src, left_on=['Origin', 'FlightDate', 'floor_informationtime'], right_on=['Origin_src', 'FlightDate_src', 'time_ceil'], how='left',suffixes=('', '_src'))
still_needed_cols = ['FlightID'] + col_names
df_rollling_avg = df_rollling_avg[still_needed_cols].copy()

for coll in col_names:
    df_rollling_avg[coll] = df_rollling_avg[coll].ffill(limit=1).fillna(0)
# drop the colums we merged with

# merge 
dataset = dataset.merge(df_rollling_avg, left_on='FlightID', right_on='FlightID', how='left')
# display the df
display(dataset.head())

# garbage collect the variables we dont need anymore
del src,   df_rollling_avg

['R_dep_cum_count_sfd', 'R_dep_cum_DepDelayMinutes_sfd', 'R_dep_avg_DepDelayMinutes_sfd', 'R_dep_count_fligts_lh', 'R_dep_avg_DepDelayMinutes_lh']


,Origin_src,time_ceil,FlightDate_src,DepDateTime,DepDelayMinutes,Cancelled,Year,R_dep_cum_count_sfd,R_dep_cum_DepDelayMinutes_sfd,R_dep_avg_DepDelayMinutes_sfd,R_dep_count_fligts_lh,R_dep_avg_DepDelayMinutes_lh,flight_count_sfd
6,ATL,2014-01-01 07:00:00,2014-01-01,2014-01-01 06:58:00,0.0,0.0,2014.0,3.0,0.0,0.000000,2.0,67.000000,1.0
7,ATL,2014-01-01 08:00:00,2014-01-01,2014-01-01 07:42:00,1.0,0.0,2014.0,8.0,3.0,0.375000,9.0,0.666667,1.0
8,ATL,2014-01-01 09:00:00,2014-01-01,2014-01-01 08:59:00,0.0,0.0,2014.0,27.0,101.0,3.740741,22.0,2.045455,1.0
9,ATL,2014-01-01 10:00:00,2014-01-01,2014-01-01 10:00:00,0.0,0.0,2014.0,55.0,152.0,2.763636,9.0,0.000000,1.0
10,ATL,2014-01-01 11:00:00,2014-01-01,2014-01-01 11:00:00,0.0,0.0,2014.0,88.0,247.0,2.806818,9.0,10.555556,1.0


,Unnamed: 0,Year,Month,DayofMonth,DayOfWeek,FlightDate,Reporting_Airline,Tail_Number,Flight_Number_Reporting_Airline,Origin,...,Information_time,floor_informationtime,FlightID,DayOfYear,route_delay,R_dep_cum_count_sfd,R_dep_cum_DepDelayMinutes_sfd,R_dep_avg_DepDelayMinutes_sfd,R_dep_count_fligts_lh,R_dep_avg_DepDelayMinutes_lh
0,0,2014,1,30,4,2014-01-30,AA,N006AA,2377,DFW,...,2014-01-30 07:40:00,2014-01-30 07:00:00,0,30,0.0,NaN,NaN,NaN,NaN,NaN
1,1,2014,1,31,5,2014-01-31,AA,N003AA,2377,DFW,...,2014-01-31 07:40:00,2014-01-31 07:00:00,1,31,9.0,NaN,NaN,NaN,NaN,NaN
2,2,2014,1,1,3,2014-01-01,AA,N002AA,2377,ICT,...,2014-01-01 09:35:00,2014-01-01 09:00:00,2,1,-7.0,NaN,NaN,NaN,NaN,NaN
3,3,2014,1,2,4,2014-01-02,AA,N002AA,2377,ICT,...,2014-01-02 09:35:00,2014-01-02 09:00:00,3,2,0.0,NaN,NaN,NaN,NaN,NaN
4,4,2014,1,3,5,2014-01-03,AA,N014AA,2377,ICT,...,2014-01-03 09:35:00,2014-01-03 09:00:00,4,3,0.0,NaN,NaN,NaN,NaN,NaN


In [34]:
# get the information for all incomming flights.

# a Arrival Date is needed. in local DESt airport time.
dataset['ArrDate'] = dataset['CRSArrDateTime'].dt.date # we can use the scheduled arrival time, as the actual arrival time is not known at the information time, and the scheduled arrival time is the best estimate we have for the actual arrival time at that moment.

arrival_origin_rolling_averages_cols =["ArrDelayMinutes","route_delay","LateAircraftDelay", "WeatherDelay","NASDelay","CarrierDelay"]
src,col_names = rolliing_day_so_far_data(dataset, "Dest", "ArrDate", "ArrDateTime", arrival_origin_rolling_averages_cols, "R_arr_origin")
display(src.head())
df_rollling_avg = dataset_airport_limit.merge(src, left_on=['Origin', 'FlightDate', 'floor_informationtime'], right_on=['Dest_src', 'ArrDate_src', 'time_ceil'], how='left',suffixes=('', '_src'))
still_needed_cols = ['FlightID'] + col_names
df_rollling_avg = df_rollling_avg[still_needed_cols].copy()

# if there was no inforamtion about the specific hour, we will use information about the hour before.
for coll in col_names:
    df_rollling_avg[coll] = df_rollling_avg[coll].ffill(limit=1).fillna(0)


dataset = dataset.merge(df_rollling_avg, left_on='FlightID', right_on='FlightID', how='left')

dataset = dataset.drop(columns=['ArrDate'])
display(dataset.head())



['R_arr_origin_cum_count_sfd', 'R_arr_origin_cum_ArrDelayMinutes_sfd', 'R_arr_origin_avg_ArrDelayMinutes_sfd', 'R_arr_origin_cum_route_delay_sfd', 'R_arr_origin_avg_route_delay_sfd', 'R_arr_origin_cum_LateAircraftDelay_sfd', 'R_arr_origin_avg_LateAircraftDelay_sfd', 'R_arr_origin_cum_WeatherDelay_sfd', 'R_arr_origin_avg_WeatherDelay_sfd', 'R_arr_origin_cum_NASDelay_sfd', 'R_arr_origin_avg_NASDelay_sfd', 'R_arr_origin_cum_CarrierDelay_sfd', 'R_arr_origin_avg_CarrierDelay_sfd', 'R_arr_origin_count_fligts_lh', 'R_arr_origin_avg_ArrDelayMinutes_lh', 'R_arr_origin_avg_route_delay_lh', 'R_arr_origin_avg_LateAircraftDelay_lh', 'R_arr_origin_avg_WeatherDelay_lh', 'R_arr_origin_avg_NASDelay_lh', 'R_arr_origin_avg_CarrierDelay_lh']


,Dest_src,time_ceil,ArrDate_src,ArrDateTime,ArrDelayMinutes,route_delay,LateAircraftDelay,WeatherDelay,NASDelay,CarrierDelay,...,R_arr_origin_cum_CarrierDelay_sfd,R_arr_origin_avg_CarrierDelay_sfd,R_arr_origin_count_fligts_lh,R_arr_origin_avg_ArrDelayMinutes_lh,R_arr_origin_avg_route_delay_lh,R_arr_origin_avg_LateAircraftDelay_lh,R_arr_origin_avg_WeatherDelay_lh,R_arr_origin_avg_NASDelay_lh,R_arr_origin_avg_CarrierDelay_lh,flight_count_sfd
1,ATL,2014-01-01 06:00:00,2014-01-01,2014-01-01 05:43:00,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.000000,14.0,40.214286,-2.071429,22.142857,0.0,1.214286,15.428571,1.0
2,ATL,2014-01-01 07:00:00,2014-01-01,2014-01-01 06:47:00,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.000000,10.0,4.500000,-1.100000,3.300000,0.0,0.000000,1.200000,1.0
3,ATL,2014-01-01 08:00:00,2014-01-01,2014-01-01 07:58:00,10.0,10.0,0.0,0.0,0.0,0.0,...,0.0,0.000000,12.0,6.666667,1.250000,3.666667,0.0,0.333333,1.166667,1.0
4,ATL,2014-01-01 09:00:00,2014-01-01,2014-01-01 08:58:00,4.0,4.0,0.0,0.0,0.0,0.0,...,16.0,0.313725,9.0,10.555556,-3.888889,4.000000,0.0,4.777778,1.777778,1.0
5,ATL,2014-01-01 10:00:00,2014-01-01,2014-01-01 09:58:00,0.0,-7.0,0.0,0.0,0.0,0.0,...,16.0,0.216216,4.0,3.500000,3.500000,0.000000,0.0,0.000000,0.000000,1.0


,Unnamed: 0,Year,Month,DayofMonth,DayOfWeek,FlightDate,Reporting_Airline,Tail_Number,Flight_Number_Reporting_Airline,Origin,...,R_arr_origin_avg_NASDelay_sfd,R_arr_origin_cum_CarrierDelay_sfd,R_arr_origin_avg_CarrierDelay_sfd,R_arr_origin_count_fligts_lh,R_arr_origin_avg_ArrDelayMinutes_lh,R_arr_origin_avg_route_delay_lh,R_arr_origin_avg_LateAircraftDelay_lh,R_arr_origin_avg_WeatherDelay_lh,R_arr_origin_avg_NASDelay_lh,R_arr_origin_avg_CarrierDelay_lh
0,0,2014,1,30,4,2014-01-30,AA,N006AA,2377,DFW,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1,2014,1,31,5,2014-01-31,AA,N003AA,2377,DFW,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2,2014,1,1,3,2014-01-01,AA,N002AA,2377,ICT,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,3,2014,1,2,4,2014-01-02,AA,N002AA,2377,ICT,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,4,2014,1,3,5,2014-01-03,AA,N014AA,2377,ICT,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [35]:
# arrival infromation for all the flight that took off at an airport, but a generall to where they landed.

# information only avalable for flights that arrived at their destination 2h before the subject flight is sheduled to take of.

# we need the Arrival Date and time in the local time of the Origin airport, for that we take the local departure time, plus the elapsed time of the flight.
dataset['ArrDateTime_OriginLocal'] = dataset['DepDateTime'] + pd.to_timedelta(dataset['ActualElapsedTime'], unit='m')
# extract the date 
dataset['ArrDate_OriginLocal'] = dataset['ArrDateTime_OriginLocal'].dt.date

from_orgin_arrived_rolling_averages_cols =["ArrDelayMinutes","route_delay","LateAircraftDelay", "WeatherDelay","NASDelay","CarrierDelay"]
src,col_names = rolliing_day_so_far_data(dataset, "Origin", "ArrDate_OriginLocal", "ArrDateTime_OriginLocal", from_orgin_arrived_rolling_averages_cols, "R_dest_arrived")
display(src.head())
df_rollling_avg = dataset_airport_limit.merge(src, left_on=['Origin', 'FlightDate', 'floor_informationtime'], right_on=['Origin_src', 'ArrDate_OriginLocal_src', 'time_ceil'], how='left',suffixes=('', '_src'))

# sort the dataset by Origin, and the Local DateTime of infromation time, and forward fill 1, for some missing hours
df_rollling_avg = df_rollling_avg.sort_values(['Origin', 'ArrDate_OriginLocal_src', 'time_ceil'])

# if there was no inforamtion about the specific hour, we will use information about the hour before.
for coll in col_names:
    df_rollling_avg[coll] = df_rollling_avg[coll].ffill(limit=1).fillna(0)

still_needed_cols = ['FlightID'] + col_names
df_rollling_avg = df_rollling_avg[still_needed_cols].copy()
# for flights where there is no available information
dataset = dataset.merge(df_rollling_avg, left_on='FlightID', right_on='FlightID', how='left')
display(dataset.head())
# drop the temporary columns created
dataset = dataset.drop(columns=['ArrDateTime_OriginLocal', 'ArrDate_OriginLocal'])

del src, df_rollling_avg,dataset_airport_limit

c:\miniconda3\envs\imgP\Lib\site-packages\pandas\core\arrays\timedeltas.py:1163: RuntimeWarning: invalid value encountered in cast
  int_data = data.astype(np.int64)


['R_dest_arrived_cum_count_sfd', 'R_dest_arrived_cum_ArrDelayMinutes_sfd', 'R_dest_arrived_avg_ArrDelayMinutes_sfd', 'R_dest_arrived_cum_route_delay_sfd', 'R_dest_arrived_avg_route_delay_sfd', 'R_dest_arrived_cum_LateAircraftDelay_sfd', 'R_dest_arrived_avg_LateAircraftDelay_sfd', 'R_dest_arrived_cum_WeatherDelay_sfd', 'R_dest_arrived_avg_WeatherDelay_sfd', 'R_dest_arrived_cum_NASDelay_sfd', 'R_dest_arrived_avg_NASDelay_sfd', 'R_dest_arrived_cum_CarrierDelay_sfd', 'R_dest_arrived_avg_CarrierDelay_sfd', 'R_dest_arrived_count_fligts_lh', 'R_dest_arrived_avg_ArrDelayMinutes_lh', 'R_dest_arrived_avg_route_delay_lh', 'R_dest_arrived_avg_LateAircraftDelay_lh', 'R_dest_arrived_avg_WeatherDelay_lh', 'R_dest_arrived_avg_NASDelay_lh', 'R_dest_arrived_avg_CarrierDelay_lh']


,Origin_src,time_ceil,ArrDate_OriginLocal_src,ArrDateTime_OriginLocal,ArrDelayMinutes,route_delay,LateAircraftDelay,WeatherDelay,NASDelay,CarrierDelay,...,R_dest_arrived_cum_CarrierDelay_sfd,R_dest_arrived_avg_CarrierDelay_sfd,R_dest_arrived_count_fligts_lh,R_dest_arrived_avg_ArrDelayMinutes_lh,R_dest_arrived_avg_route_delay_lh,R_dest_arrived_avg_LateAircraftDelay_lh,R_dest_arrived_avg_WeatherDelay_lh,R_dest_arrived_avg_NASDelay_lh,R_dest_arrived_avg_CarrierDelay_lh,flight_count_sfd
5,ATL,2014-01-01 09:00:00,2014-01-01,2014-01-01 08:47:00,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.000000,12.0,7.916667,6.250000,0.000000,0.0,4.833333,1.250000,1.0
6,ATL,2014-01-01 10:00:00,2014-01-01,2014-01-01 09:51:00,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.000000,19.0,4.526316,4.210526,0.000000,0.0,2.473684,0.210526,1.0
7,ATL,2014-01-01 11:00:00,2014-01-01,2014-01-01 10:55:00,0.0,0.0,0.0,0.0,0.0,0.0,...,77.0,2.851852,11.0,12.454545,0.000000,0.727273,0.0,2.272727,8.818182,1.0
8,ATL,2014-01-01 12:00:00,2014-01-01,2014-01-01 11:55:00,0.0,0.0,0.0,0.0,0.0,0.0,...,77.0,1.833333,10.0,1.800000,-6.600000,0.000000,0.0,0.000000,0.000000,1.0
9,ATL,2014-01-01 13:00:00,2014-01-01,2014-01-01 12:59:00,0.0,0.0,0.0,0.0,0.0,0.0,...,144.0,2.181818,3.0,38.000000,0.333333,0.000000,0.0,1.000000,37.000000,1.0


,Unnamed: 0,Year,Month,DayofMonth,DayOfWeek,FlightDate,Reporting_Airline,Tail_Number,Flight_Number_Reporting_Airline,Origin,...,R_dest_arrived_avg_NASDelay_sfd,R_dest_arrived_cum_CarrierDelay_sfd,R_dest_arrived_avg_CarrierDelay_sfd,R_dest_arrived_count_fligts_lh,R_dest_arrived_avg_ArrDelayMinutes_lh,R_dest_arrived_avg_route_delay_lh,R_dest_arrived_avg_LateAircraftDelay_lh,R_dest_arrived_avg_WeatherDelay_lh,R_dest_arrived_avg_NASDelay_lh,R_dest_arrived_avg_CarrierDelay_lh
0,0,2014,1,30,4,2014-01-30,AA,N006AA,2377,DFW,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1,2014,1,31,5,2014-01-31,AA,N003AA,2377,DFW,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2,2014,1,1,3,2014-01-01,AA,N002AA,2377,ICT,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,3,2014,1,2,4,2014-01-02,AA,N002AA,2377,ICT,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,4,2014,1,3,5,2014-01-03,AA,N014AA,2377,ICT,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### 3. Turnaround time
    The Trounaround time measueres the time form the Arrival of the Previous flight until the departure of the next (subject)
    aditionally other features,regarding the previous flight of the aircraft are collected and used.

0. "Flights_before_today" ->     How manny flights if the same plane have already been scheduled for today
1. "total_Flights_scheduled_today" ->    how manny in totoal this aircaft has scheduled for today
2. "CRSTurnaroundTime" ->    Sheduled turnaorund time 
3. "has_prev_flight" ->  If there was a certified Pervious flight at all, which fits the criteria
4. "is_first_flight" ->  if there is any mention at all of this aircraft before
5. "Time_since_last_certified_record" ->     the last moment the aircraft was recoreded somewherer beofre nextt flight (if not normal previous flight)
6. "Same_day_previous_flight" ->     if the previous flight detected was on the same FlightDate, as next.
7. "is_Return_flight" ->     if the previous flight came form the airport the next flight goes to agian.
8. "Prev_flight_DelayMinutes" ->     the so far known Delay of the previous flight (depends on where the previous flight is 2h before next flight)
9. "Prev_flight_has_departed" ->     if the previous flight has departed its Orgin airport yet
10. "Airplane_already_at_airport" ->     if the previous flight already arrived at the destination airport (and therfore the origin of the next flight)
11. "Expected_Tournaround_time" ->   Scheduled Turnaround time - the so far known Delay of the previous flight.
12. [Delay Reasons of previous flight] ->    multiple features, that have the number of minutes the previous flight was delayed because of if that information is avaliabel (if the previous flight is already at its destiantion)



In [ ]:


# we now start computing the Turnaround time
# we sort the dataset by tail number and flight date, to make it easier to find the previous flight
dataset = dataset.sort_values(by=['Tail_Number', 'CRSDepDateTime_UTC'])




#  === Number fo flights of the same plane on the same day ===
# How manny flights if the same plane have already been scheduled for today
dataset["Flights_before_today"] = dataset.groupby(['Tail_Number', 'FlightDate']).cumcount()
#  how manny in totoal 
dataset["total_Flights_scheduled_today"] = dataset.groupby(['Tail_Number', 'FlightDate'])['FlightID'].transform('count')
# =====

# ==== Tournaround time and the previous flight information ====
# Finding the previous flight for each aircraft by shifting rows
prev_cols = {
    'FlightID': 'PreviousFlightId',
    'Dest': 'PreviousFlightDest',
    'Origin': 'PreviousFlightOrigin',
    "FlightDate": "PreviousFlightDate",
    'Reporting_Airline': 'PreviousFlightAirline',
    'Tail_Number': 'PreviousFlightTailNumber',
    'Cancelled': 'PreviousFlightCancelled',
    'Diverted': 'PreviousFlightDiverted',
    'CRSArrDateTime_UTC': 'CRSPreviousFlightArrDateTime_UTC',
    'ArrDateTime_UTC': 'PreviousFlightArrDateTime_UTC',
    'CRSDepDateTime_UTC': 'CRSPreviousFlightDepDateTime_UTC',
    'DepDateTime_UTC': 'PreviousFlightDepDateTime_UTC',
    'DepDelayMinutes': 'PreviousFlightDepDelayMinutes',
    'ArrDelayMinutes': 'PreviousFlightArrDelayMinutes'
}
# get the previous flight information by shifing the rows.
for col, new_col in prev_cols.items():
    dataset[new_col] = dataset[col].shift(1)

# --- Validate if the prev Row is really the previous flight of that aircraft.
# -- Creatign a mask to check if the row before is the preivous flight, in 3 steps:

# 1. CHeck if the Tail numbers and airline match, as we sorted for this so it should be the same one, or we have the first appearance of of this tail number
found_aircraft = (dataset['Tail_Number'] == dataset['PreviousFlightTailNumber']) & (dataset['Reporting_Airline'] == dataset['PreviousFlightAirline'])
# if the Tail number does not match we have the first appearance of this tail number, so we set the mask to true, to not lose this information
dataset['FirstFlightRecord'] = (~found_aircraft).astype(int) # if the mask is false, it is the first flight record of this tail number, so we set it to 1, otherwise 0.


# 2. check if the flight before also was not diverted or cancelled
last_flight_happend = found_aircraft & (dataset['PreviousFlightCancelled'] == 0) & (dataset['PreviousFlightDiverted'] == 0) 


# 3. check if the departure airport of the current flight matches the arrival airport of the previous flight, as this is also a strong indicator that it is the previous flight, as most flights do not change their route that often.
has_prev_flight= last_flight_happend & (dataset['Origin'] == dataset['PreviousFlightDest'])

        # when did we last see this plane?, if ther is no proper turnaround.
dataset["Time_since_last_certified_record"] = ((dataset['CRSDepDateTime_UTC'] - dataset['CRSPreviousFlightArrDateTime_UTC']).dt.total_seconds() / 60).where(has_prev_flight &~has_prev_flight, other=0)
# if the mask3 is true, we have a valid previous flight, otherwise not, so we set it to 1 or 0.
dataset["has_prev_flight"] = has_prev_flight.astype(int) 

# 4. we remove the infomration about the prev flight to all flights which do not pass all 3 checks
def mask_previous_flight_info(col):
    return dataset[col].where(has_prev_flight, other=np.nan)
# run this function for all the columns we need for the previous flight information

previous_flight_time_info_cols = ['PreviousFlightId', 'CRSPreviousFlightArrDateTime_UTC', 'PreviousFlightArrDateTime_UTC',
                                   'CRSPreviousFlightDepDateTime_UTC', 'PreviousFlightDepDateTime_UTC', 'PreviousFlightDepDelayMinutes', 'PreviousFlightArrDelayMinutes']
for col in previous_flight_time_info_cols:
    dataset[col] = mask_previous_flight_info(col)



# Aditional feature to check if the previous flight was on the same day, as this is also a strong indicator that it is the previous flight, as most flights do not have a turnaround time of more than 24h, and if it is the same day it is more likely to be the previous flight, than if it is not the same day.
dataset["Same_day_previous_flight"] = ((dataset['PreviousFlightDate'] == dataset['FlightDate'])).astype(int)
# feature to check if this is a return flight, for the previous flight.
dataset["is_Return_flight"] = ((dataset['Origin'] == dataset['PreviousFlightDest']) & (dataset['Dest'] == dataset['PreviousFlightOrigin'])).astype(int)

# drop Columns we do not need anymore, about the previous flight
dataset = dataset.drop(columns=[ 'PreviousFlightDest', 'PreviousFlightOrigin', 'PreviousFlightAirline', 'PreviousFlightTailNumber', 'PreviousFlightCancelled', 'PreviousFlightDiverted', 'PreviousFlightDate'])








In [ ]:
# here we drop the Data of the previus Year.


In [ ]:


dataset['CRSTurnaroundTime'] = (dataset['CRSDepDateTime_UTC'] - dataset['CRSPreviousFlightArrDateTime_UTC']).dt.total_seconds() / 60
mean_turnaround_time = dataset['CRSTurnaroundTime'].mean()
# fill the nan with the Mean.
dataset['CRSTurnaroundTime'] = dataset['CRSTurnaroundTime'].fillna(mean_turnaround_time)

# opportunity to maybe add furhter feature, that are regarding the last flight of an aircraft


# === IF flight is already confirmed at airport 2h Before the sceduled flight ===

# Localize knownWeatherDateTime_UTC to UTC to make it comparable

# if ther is no previous flight, we dont know it so we dont know if the airplane is at the airport already

# === information if the previous flight has a delay already, that we can know of, bacuse the departure was more than 2h ago
# the min departure delay of the Previous flight.
# there are 3 cases for the calululation of this feature:
# 1. the Plane was not supposed to Depart already, 2h before the subject flight (informationtime) -> the previous delay is 0
# 2. the plane was supposed to depart already and has -> Departure DelayMinutes
# 3. was supposed to but hasnt yet, than the diffrence from now to the planned DepTime

print("start")
# temporarly get the timezone info to the information time, to compare it to the others.
dataset["Information_time_UTC"] = dataset["Information_time_UTC"].dt.tz_localize('UTC')

Prev_should_have_Departed = (dataset["Information_time_UTC"]>= dataset["CRSPreviousFlightDepDateTime_UTC"])   # for 1
Prev_has_departed_mask = (dataset["Information_time_UTC"] >= dataset["PreviousFlightDepDateTime_UTC"])   # for 2
Prev_should_have_but_not_departed_mask = (Prev_should_have_Departed & (~Prev_has_departed_mask)) # for 3
Prev_has_arrived_mask = (dataset["Information_time_UTC"] >= dataset["PreviousFlightArrDateTime_UTC"])  # for 4
Prev_should_have_arrived = (dataset["Information_time_UTC"] >= dataset["CRSPreviousFlightArrDateTime_UTC"]) & (~Prev_has_arrived_mask) # for 5

# 1.  where it shoudl have departed
dataset["Prev_flight_DelayMinutes"] = 0 # redunent as line would be np.where(~Prev_should_have_Departed, 0, 0)
# 2. where has departed we take the Dep Delay 
dataset["Prev_flight_DelayMinutes"] = np.where(Prev_has_departed_mask, dataset["PreviousFlightDepDelayMinutes"], dataset["Prev_flight_DelayMinutes"])
dataset["Prev_flight_has_departed"] = Prev_has_departed_mask.astype(int) # 1 if the plane has departed, 0 if not, this is also a feature in itself, as if the plane has departed it is more likely that it is at the airport already, than if it has not departed yet.
# 3. where should but has not
dataset["Prev_flight_DelayMinutes"] = np.where(Prev_should_have_but_not_departed_mask, (dataset["Information_time_UTC"] - dataset["PreviousFlightDepDateTime_UTC"]).dt.total_seconds() / 60, dataset["Prev_flight_DelayMinutes"])
# 4. Check if the plane has arrived already
dataset["Prev_flight_DelayMinutes"] = np.where(Prev_has_arrived_mask, dataset["PreviousFlightArrDelayMinutes"], dataset["Prev_flight_DelayMinutes"])
# make the fact that the plane is aleady at the airport a feature in it self
dataset["Airplane_already_at_airport"] = np.where(Prev_has_arrived_mask, 1.0, 0.0) # 1 if plane has arrived, 0 if not
dataset.loc[(dataset['has_prev_flight']==0),["Airplane_already_at_airport"]] = 0.5 # 0.5 if we do not know becuase there is no previous flight.

# 5. check if the plane should have arrived but has not, than we take the difference from now to the planned ArrTime as delay
dataset["Prev_flight_DelayMinutes"] = np.where((Prev_should_have_arrived & (~Prev_has_arrived_mask)), (dataset["Information_time_UTC"] - dataset["CRSPreviousFlightArrDateTime_UTC"]).dt.total_seconds() / 60, dataset["Prev_flight_DelayMinutes"])
# to make sure, we will set the delay to 0 if there is no previous flight
dataset["Prev_flight_DelayMinutes"] = np.where(has_prev_flight, dataset["Prev_flight_DelayMinutes"], 0)
print("finished")

# convert to an Expected Turnaround time, by reducing the delay of the previous flight from the planned turnaround time, as if the previous flight has a delay, it is likely that the turnaround time will be shorter, as the plane is already at the airport and can be prepared for the next flight.
dataset["Expected_Tournaround_time"] = dataset["CRSTurnaroundTime"] - dataset["Prev_flight_DelayMinutes"]
dataset["Expected_Tournaround_time"] = dataset["Expected_Tournaround_time"].fillna(dataset["CRSTurnaroundTime"])

# finished  the previus flight data, drop the columns we do not need anymore
# the prebious_flight_time_info_cols are the remainig ones, that need to be dropped


In [ ]:
# additional information we can provide, if the plane is already at the airport, aout the delay reasons

# merge the delay reasosn of the previous flight, if it is already at the aiport, to the current one
dataset = dataset.merge(dataset[["FlightID", "LateAircraftDelay", "WeatherDelay","NASDelay","CarrierDelay"]], left_on='PreviousFlightId', right_on='FlightID', how='left', suffixes=('', '_prev_flight_delay_info'))
# remove the flights where the plane is not at the airport already, as we do not know the delay reasons for these flights, as the previous flight has not arrived yet, so we set it to 0, to not lose this information
prev_delay_reason_cols =["LateAircraftDelay_prev_flight_delay_info", "WeatherDelay_prev_flight_delay_info","NASDelay_prev_flight_delay_info","CarrierDelay_prev_flight_delay_info"]
dataset.loc[dataset['Airplane_already_at_airport'] == 0, prev_delay_reason_cols ] = 0
#fill na with 0
dataset[prev_delay_reason_cols] = dataset[prev_delay_reason_cols].fillna(0)
# drop the Flight ID of th previous flight we jsut merged again
dataset = dataset.drop(columns="FlightID_prev_flight_delay_info")

In [ ]:
dataset = dataset[dataset['Year'] >= min_year]

dataset = dataset.drop(columns=previous_flight_time_info_cols)
print(f"has_prev_flight: {dataset['has_prev_flight'].mean()*100:.1f}% of flights have previous flight data")

In [ ]:
# how manny nan values are in the CRSTurnaroundTime column
num_nan = dataset['CRSTurnaroundTime'].isna().sum()
print("Number of NaN values in the CRSTurnaroundTime column: ", num_nan)

# how manny times the tailnumber is missing in the dataset
num_nan_tailnumber = dataset['Tail_Number'].isna().sum()
print("Number of NaN values in the Tail_Number column: ", num_nan_tailnumber)
display(dataset)
#remove the timezoe from the information time again


In [ ]:
dataset["Information_time_UTC"] = dataset["Information_time_UTC"].dt.tz_localize(None)

In [ ]:
# reduce the Dataset, to the things we actually want to train with.
dataset = dataset[~dataset['CRSElapsedTime'].isna()]
# we finally remove all the flights that do not land and depart from an airport in the List.
dataset = dataset[dataset["Origin"].isin(AIRPORT_LIMIT_LIST) & dataset["Dest"].isin(AIRPORT_LIMIT_LIST)]
dataset = dataset[~((dataset['Diverted'] == 1) | (dataset['Cancelled'] == 1))]
num_nan_tailnumber = dataset['Tail_Number'].isna().sum()
print("Number of NaN values in the Tail_Number column without cancelled/diverted flights: ", num_nan_tailnumber)

#### Bejond this point only the Entries intended for training are in the Dataset

> this reduces RAM load, but means that the Possibility for new fetaures is limited, beyond this point.

### 4. Holiday features

In [ ]:
# Using holiday library

# Generate holiday dates for all years in the dataset
years_in_data = range(int(dataset['Year'].min()), int(dataset['Year'].max()) + 1)
us_holiday_dates = holidays.US(years=years_in_data)
# Add Christmas Eve and New Year's Eve (not in the official US holidays list)
for y in years_in_data:
    us_holiday_dates[pd.Timestamp(y, 12, 24)] = "Christmas Eve"
    us_holiday_dates[pd.Timestamp(y, 12, 31)] = "New Year's Eve"



# holiday_df = pd.DataFrame({'holiday_date': sorted(us_holiday_dates.keys())})
# holiday_df['holiday_date'] = pd.to_datetime(holiday_df['holiday_date']).dt.normalize()

holiday_dates_set = set(pd.to_datetime(list(us_holiday_dates.keys())).normalize())
print(f"Total holiday dates generated: {len(holiday_dates_set)} across {len(list(years_in_data))} years")

# dataset = dataset.sort_values('FlightDate')
dataset['FlightDate'] = pd.to_datetime(dataset['FlightDate']).dt.normalize()

# IsHoliday: binary flag
dataset['IsHoliday'] = dataset['FlightDate'].dt.normalize().isin(holiday_dates_set).astype(int)
print(f"Flights on holidays: {dataset['IsHoliday'].sum():,} ({dataset['IsHoliday'].mean()*100:.1f}%)")

# DaysToNearestHoliday: distance in days to the closest holiday
holidays_arr = np.array(sorted(holiday_dates_set), dtype='datetime64[D]')
flight_dates_arr = dataset['FlightDate'].dt.normalize().values.astype('datetime64[D]')
# find fitting holidays to flightdates
idx = np.searchsorted(holidays_arr, flight_dates_arr)
# handle last and first holdays
idx_prev = np.clip(idx - 1, 0, len(holidays_arr) - 1)
idx_next = np.clip(idx, 0, len(holidays_arr) - 1)

# calculate distances
dist_prev = np.abs((flight_dates_arr - holidays_arr[idx_prev]).astype('timedelta64[D]').astype(int))
dist_next = np.abs((holidays_arr[idx_next] - flight_dates_arr).astype('timedelta64[D]').astype(int))


# extract teh min
dataset['DaysToNearestHoliday'] = np.minimum(dist_prev, dist_next)

print(f"DaysToNearestHoliday: mean={dataset['DaysToNearestHoliday'].mean():.1f}, max={dataset['DaysToNearestHoliday'].max()}")

### 5. Weather data api features

In [ ]:
# load the airoorts dataset
airports = storage.read_csv('data/external/airports_with_runway_info.csv')
# only keep the airports in the Airport List.
airports = airports[airports['iata_code'].isin(AIRPORT_LIMIT_LIST)]

# merge the airports dataset with the flights dataset to get the timezone information for the departure and arrival airports
display(airports)
keep_cols = ['iata_code', 'type','scheduled_service','num_runways','most_common_surface','avg_runway_length','has_lighted_runways' ]
weather_cols = ['iata_code']
# get the colum index number of 'airport_station'
airport_station_index = airports.columns.get_loc('airport_station')
# add all columns from the index of 'airport_station' to the end of the dataframe to the list of columns to keep
weather_cols += airports.columns[airport_station_index:].tolist()

In [ ]:
ms.config.block_large_requests = False
airports_weather = airports[weather_cols]
# grop the flights dataset, and search for the fist and last flight date for each airport, and merge this information with the airports dataset
airport_flight_dates = dataset.groupby('Origin')['floor_informationtime_UTC'].agg(['min', 'max']).reset_index()
# also do the same for the destination airports
airport_flight_dates_dest = dataset.groupby('Dest')['floor_informationtime_UTC'].agg(['min', 'max']).reset_index()
# get the min out of both min dates, and the max out of both max dates, to get the date range for which we need weather data for each airport
airport_flight_dates = airport_flight_dates.merge(airport_flight_dates_dest, left_on='Origin', right_on='Dest', how='outer', suffixes=('_origin', '_dest'))

# subtract 5h from the min date, and add 5 hours to the max date, to get the date range for which we need weather data
airport_flight_dates['min'] = (airport_flight_dates[['min_origin', 'min_dest']].min(axis=1) - pd.Timedelta(hours=5)).dt.tz_localize(None) # get additional 5H and remove timezone info
airport_flight_dates['max'] = (airport_flight_dates[['max_origin', 'max_dest']].max(axis=1) + pd.Timedelta(hours=5)).dt.tz_localize(None) # get additional 5H and remove timezone info
airport_flight_dates = airport_flight_dates[['Origin', 'min', 'max']].rename(columns={'Origin': 'iata_code'})

# merge the airport flight dates with the airports weather dataset to get the date range for which we need weather data for each airport
airport_weather_dates = airport_flight_dates.merge(airports_weather, on='iata_code', how='inner')
# check if the length of all three datasets is the same, if not there are some airports for which we do not have weather data, and we need to drop them from the flights dataset
print(f"Length of airport_flight_dates: {len(airport_flight_dates)}")
print(f"Length of airports_weather: {len(airports_weather)}")
print(f"Length of airport_weather_dates: {len(airport_weather_dates)}")

In [ ]:
weather_params = ["temp","prcp","wspd","rhum"]
weather_data = pd.DataFrame()
print("Getting weather data for each airport, this may take a while...")

# function to fill the Weather Dataframe , with weatehr data.
def fill_weather_data(data, airport, date_range_length):
    data_length = len(data)
    # if the data is less than 95% complete, we try to fill it up with the data from the other near stations 1-3
    if data_length < date_range_length:
        print("filling up")
        # get data from another station
        for i in range(1,4):
            if airport[f'closest_station_{i}'] and airport[f'closest_station_{i}_distance'] < 50_000:
                data1 = ms.hourly(airport[f'closest_station_{i}'], airport['min'], airport['max'], parameters=weather_params)
                data1 = data1.fetch()
                if data1 is None or data1.empty:
                    continue
                data1["airport"] = airport["iata_code"]
                data1["timestamp"] = data1.index
                # match the data from the two stations and fill the missing rows in the first dataset with the values from the second dataset
                data = data.merge(data1, on=["timestamp", "airport"], how="outer", suffixes=("", "_1"))
                # fill every value on the first dataset with the value from the second dataset if it is missing in the first dataset
                for param in weather_params:
                    data[param] = data[param].fillna(data[f"{param}_1"])
                    data = data.drop(columns=[f"{param}_1"])
                if len(data) < date_range_length:
                    break
        print('finished filling up')
    return data

In [ ]:
# Collecting the Weather Data.
total_added_rows = 0
for index, airport in airport_weather_dates.iterrows():
    data_range_length = (airport['max'] - airport['min']).total_seconds() / 3600 # the number of Hours from max to min (how manny entries the Weather Data df will need)
    iata_code = airport['iata_code']

    data = None
    # base case: we have an aiport station, with weather data, which is mostly complete
    if airport['airport_station'] is not np.nan and airport['airport_data_length_code'] is not np.nan:
        #  get the Weather Data for every Hour
        data = ms.hourly(airport['airport_station'], airport['min'], airport['max'], parameters=weather_params)
        data = data.fetch()
        # if the Data is somehow empty, we create an empty df
        if data is None or data.empty:
            data = pd.DataFrame(columns=["timestamp", "airport"]+weather_params)
        # add columns to match the data on airport and hour later.
        data["airport"] = iata_code
        data["timestamp"] = data.index # the index is also the time, so this is essestially a copy

        # get the length of the data, and if it is less than 80% of the date range, we fill it up with the data from the other near stations 1-3
        data= fill_weather_data(data, airport, data_range_length) #calling the function
    else:
        data = pd.DataFrame(columns=["timestamp", "airport"]+weather_params)
        # if we do not have an airport station, we try to fill the data with the data from the other near stations 1-3
        data = fill_weather_data(data, airport, data_range_length)
    # now we have collected The Weather data



    # ---- This Block Makes shure for every Needed Hour there is a corresponding, row, even though it might be NULL for now
    # -> so we can performe a missing value operations later
    if len(data) <= data_range_length: # check if the data is incomplete, and there are Gaps
        #  we add rows, for the missing hours
        print(f" {iata_code} is incomplete. ({index+1}/{len(airport_weather_dates)}) --- {len(data)} rows ")
        # get the range of the data
        previ_length = len(data) #prev Or old Length.
        data_range = (airport['min'], airport['max'])
        # create a complete range of timestamps for the date range to make sure the entries for every hour exisit, even though might contain nothing
        complete_range = pd.date_range(start=data_range[0], end=data_range[1], freq='h')
        # make it a dataframe
        complete_range = pd.DataFrame(complete_range, columns=['timestamp'])
        complete_range['airport'] = airport['iata_code']
        # merge the complete range with the data to get the missing timestamps
        data = complete_range.merge(data, on=['timestamp', 'airport'], how='outer') # guarantee that for every hour, needed, there is an Hour.
        new_length = len(data)
        # print(f"added {new_length - previ_length} rows to the data for airport {iata_code}")
        total_added_rows += new_length - previ_length #track the added rows.
    else: # the Data is complete, and there are no holes.
        # print(f"{iata_code} is complete. ({index+1}/{len(airport_weather_dates)}) --- {len(data)} rows")
        pass



    # --- perform filling Nans. Standart method, is forward fill ( filling with the value of the last valid entry, with max 2 entries before)
    for param in weather_params:
        # try filling single nans with the previous value, if that is not also nan, we leave it as nan
        data[param] = data[param].ffill(limit=2)
        if data[param].isna().sum() / len(data) > 0.5:
            print(f"Warning: More than 50% of the values in the {param} column are missing for airport {iata_code}. Consider filling them with a constant value or using a different imputation method.")
            data[param] = data[param].fillna(0)
        else:
            data[param] = data[param].fillna(0)

    weather_data = pd.concat([weather_data, data], ignore_index=True)

In [ ]:
print(f"Total rows added: {total_added_rows}")
print(len(dataset))
display(weather_data)
#-  Merge the Weather Data onto the Departure and arrival Airport.
dataset = dataset.merge(weather_data, left_on=["Origin", "floor_informationtime_UTC"], right_on=["airport", "timestamp"], how="left", suffixes=("", "_DEP"))
dataset = dataset.merge(weather_data, left_on=["Dest", "floor_informationtime_UTC"], right_on=["airport", "timestamp"], how="left", suffixes=("_DEP", "_ARR"))

display(dataset)

In [ ]:
# get all the rows where the timestamp_Dep or timestamp_ARR is missing
missing_arr = dataset[dataset['timestamp_ARR'].isna()]
missing_dep = dataset[dataset['timestamp_DEP'].isna()]
# union of both missing datasets
missing = pd.concat([missing_arr, missing_dep])
# throw out duplicates
missing = missing.drop_duplicates()
display(missing)
# drop the rows where the timestamp_Dep is missing, as we can not use them for training
dataset = dataset[~dataset['timestamp_ARR'].isna()]
dataset = dataset[~dataset['timestamp_DEP'].isna()]

# drop the temporary collumns
dataset = dataset.drop(columns=['timestamp_DEP', 'timestamp_ARR','airport_DEP','airport_ARR'])

# convert the temp, prcp and wspd columns to numeric, as they are currently object due to the nans
for col in weather_params:
    dataset[f"{col}_ARR"] = pd.to_numeric(dataset[f"{col}_ARR"], errors='coerce')
    dataset[f"{col}_DEP"] = pd.to_numeric(dataset[f"{col}_DEP"], errors='coerce')
del weather_data, airport_weather_dates, airports_weather, airports, weather_params


### 7. departure hour feature

In [ ]:
# === Static Time Feature ===
# dep_hour: captures peak-time delay patterns (raw CRSDepDateTime can't be fed to models)
dataset["dep_hour"] = dataset["CRSDepDateTime"].dt.hour

print(f"dep_hour: nunique={dataset['dep_hour'].nunique()}, null={dataset['dep_hour'].isna().sum()}")

### 8. Rolling avg features

In [ ]:
dataset = dataset.sort_values("CRSDepDateTime_UTC").reset_index(drop=True)
# dict for abriviations for target values

def compute_hist_features(df, group_col,target_col, prefix,rolling_windows=[7,30,-1]):
    """Compute expanding + 7d + 30d rolling means for a given grouping."""

    daily_agg = (df.groupby(["FlightDate", group_col])[target_col]
                 .mean().reset_index()
                 .rename(columns={target_col: f"{prefix}_daily_delay"}))
    cols_tomerge = ["FlightDate", group_col]

    daily_agg = daily_agg.sort_values("FlightDate")
    col = f"{prefix}_daily_delay"
    for window in rolling_windows:
         if window == -1:
             daily_agg[f"hist_{prefix}_delay"] = (
                 daily_agg.groupby(group_col)[col]
                 .transform(lambda x: x.shift(1).expanding().mean()))
             cols_tomerge.append(f"hist_{prefix}_delay")
         else:
             daily_agg[f"hist_{prefix}_delay_{window}d"] = (
                 daily_agg.groupby(group_col)[col]
                 .transform(lambda x: x.shift(1).rolling(window, min_periods=1).mean()))
             cols_tomerge.append(f"hd_{prefix}__{window}d")
    return_df =df.merge(daily_agg[cols_tomerge], on=["FlightDate", group_col], how="left")
    return return_df



def compute_hist_features(df, daily_agg, group_col, prefix):
    """Compute expanding + 7d + 30d rolling means for a given grouping."""
    daily_agg = daily_agg.sort_values("FlightDate")
    col = f"{prefix}_daily_delay"

    daily_agg[f"hist_{prefix}_delay"] = (
        daily_agg.groupby(group_col)[col]
        .transform(lambda x: x.shift(1).expanding().mean()))
    daily_agg[f"hist_{prefix}_delay_7d"] = (
        daily_agg.groupby(group_col)[col]
        .transform(lambda x: x.shift(1).rolling(7, min_periods=1).mean()))
    daily_agg[f"hist_{prefix}_delay_30d"] = (
        daily_agg.groupby(group_col)[col]
        .transform(lambda x: x.shift(1).rolling(30, min_periods=1).mean()))

    merge_cols = ["FlightDate", group_col,
                  f"hist_{prefix}_delay", f"hist_{prefix}_delay_7d", f"hist_{prefix}_delay_30d"]
    return df.merge(daily_agg[merge_cols], on=["FlightDate", group_col], how="left")


In [ ]:
# === Historical Rolling Delay Features ===
# Expanding mean + 7-day + 30-day rolling mean of ArrDelayMinutes,
# grouped by airline, origin, and destination.
# shift(1) excludes the current day to prevent leakage.
# (Route grouping omitted — redundant with separate origin + dest features)

dataset = dataset.sort_values("CRSDepDateTime_UTC").reset_index(drop=True)


# Per airline
daily_airline = (dataset.groupby(["FlightDate", "Reporting_Airline"])["ArrDelayMinutes"]
                 .mean().reset_index()
                 .rename(columns={"ArrDelayMinutes": "airline_daily_delay"}))
dataset = compute_hist_features(dataset, daily_airline, "Reporting_Airline", "airline")

# Per origin airport
daily_origin = (dataset.groupby(["FlightDate", "Origin"])["ArrDelayMinutes"]
                .mean().reset_index()
                .rename(columns={"ArrDelayMinutes": "origin_daily_delay"}))
dataset = compute_hist_features(dataset, daily_origin, "Origin", "origin")

# Per destination airport
daily_dest = (dataset.groupby(["FlightDate", "Dest"])["ArrDelayMinutes"]
              .mean().reset_index()
              .rename(columns={"ArrDelayMinutes": "dest_daily_delay"}))
dataset = compute_hist_features(dataset, daily_dest, "Dest", "dest")

hist_features = [
    "hist_airline_delay", "hist_airline_delay_7d", "hist_airline_delay_30d",
    "hist_origin_delay", "hist_origin_delay_7d", "hist_origin_delay_30d",
    "hist_dest_delay", "hist_dest_delay_7d", "hist_dest_delay_30d",
]

# Fill NaN (first days have no history) with 0
for f in hist_features:
    dataset[f] = dataset[f].fillna(0)

print("Historical rolling features:")
for f in hist_features:
    print(f"  {f}: null={dataset[f].isna().sum():,}, mean={dataset[f].mean():.2f}")

del daily_airline, daily_origin, daily_dest

### 8. Lag features

In [ ]:
# === Lag-1 / Lag-7 Features (from autocorrelation analysis) ===
# Yesterday's and last-week's mean delay as direct features.
# Lag-1 autocorrelation was ~0.53, lag-7 ~0.21 (weekly cycle).
# (Route grouping omitted — redundant with separate origin + dest features)

def add_lag_features(df, group_col, prefix):
    """Add lag-1 (yesterday) and lag-7 (last week) delay features for a grouping."""
    daily = (df.groupby(["FlightDate", group_col])["ArrDelayMinutes"]
             .mean().reset_index()
             .rename(columns={"ArrDelayMinutes": f"{prefix}_daily_mean"}))
    daily = daily.sort_values("FlightDate")

    daily[f"{prefix}_yesterday_delay"] = (
        daily.groupby(group_col)[f"{prefix}_daily_mean"].shift(1))
    daily[f"{prefix}_lastweek_delay"] = (
        daily.groupby(group_col)[f"{prefix}_daily_mean"].shift(7))

    merge_cols = ["FlightDate", group_col,
                  f"{prefix}_yesterday_delay", f"{prefix}_lastweek_delay"]
    return df.merge(daily[merge_cols], on=["FlightDate", group_col], how="left")

# Per origin airport
dataset = add_lag_features(dataset, "Origin", "origin")
# Per destination airport
dataset = add_lag_features(dataset, "Dest", "dest")
# Per airline
dataset = add_lag_features(dataset, "Reporting_Airline", "airline")

# Global lags (no grouping — overall system delay)
daily_global = (dataset.groupby("FlightDate")["ArrDelayMinutes"]
                .mean().reset_index()
                .rename(columns={"ArrDelayMinutes": "global_daily_mean"}))
daily_global = daily_global.sort_values("FlightDate")
daily_global["global_yesterday_delay"] = daily_global["global_daily_mean"].shift(1)
daily_global["global_lastweek_delay"] = daily_global["global_daily_mean"].shift(7)
dataset = dataset.merge(daily_global[["FlightDate", "global_yesterday_delay", "global_lastweek_delay"]],
                        on="FlightDate", how="left")

lag_features = [
    "origin_yesterday_delay", "origin_lastweek_delay",
    "dest_yesterday_delay", "dest_lastweek_delay",
    "airline_yesterday_delay", "airline_lastweek_delay",
    "global_yesterday_delay", "global_lastweek_delay",
]

# Fill NaN (first days have no prior data) with 0
for f in lag_features:
    dataset[f] = dataset[f].fillna(0)

print("Lag features:")
for f in lag_features:
    print(f"  {f}: null={dataset[f].isna().sum():,}, mean={dataset[f].mean():.2f}")

del daily_global

# Drop and save

In [ ]:
# Collumns that are irrelavnt or contain leakage information, which can not be used in the model, that can be dropped:

# these contain information about the actual delay, not avaliabe at time of Prediction
target_like = ['DepDelay','DepDelayMinutes','ActualElapsedTime', 'Diverted', 'Cancelled','ArrTime','DepTime','ArrDateTime','DepDateTime','AirTime',
               'NASDelay','SecurityDelay','LateAircraftDelay','CarrierDelay','WeatherDelay','ArrDateTime_UTC','ArrDelay']
# Not relevant to training
timezone_cols = ['TZ_Origin', 'TZ_Dest']
# UTC Data, is irrelevant, and for Sheduled times, there are CRSDepDateTime columns still in the Dataframe.
timestamp_cols = ['DepDateTime_UTC', 'CRSArrDateTime_UTC', 'CRSDepTime', 'CRSArrTime','FlightDate']

# drop against overfitting
overtting_columns = ['FlightID','Tail_Number','Flight_Number_Reporting_Airline']

all_to_drop = target_like + timezone_cols + timestamp_cols + overtting_columns

In [ ]:
# create a second table, which just contains info about the features present in the dataset.
df_columns = pd.DataFrame({
    'data_type': dataset.dtypes,
    'unique_values': dataset.nunique(),
    'nan_values': dataset.isna().sum()
})

# 2. Add the column_name and Dropped status
# .index refers to the column names of 'dataset'
df_columns['column_name'] = df_columns.index
df_columns['Dropped'] = df_columns['column_name'].isin(all_to_drop)
# Nan Values
df_columns['nan_values'] = dataset.isna().sum()

df_columns["Category"] ="Undefined"
# traget_like collumns
df_columns.loc[df_columns['column_name']=="ArrDelayMinutes", 'Category'] = "Target"
df_columns.loc[df_columns['column_name'].isin(target_like), 'Category'] = "Target Like"


# for some Collumns we can write a Category to explain what they represent:
# 1. Flight Information, liek airports, airlines, tailnumbers and similar
flight_info_cols = ["Origin", "Dest", "Reporting_Airline", "Tail_Number", "Flight_Number_Reporting_Airline","FlightID"]
df_columns.loc[df_columns['column_name'].isin(flight_info_cols), 'Category'] = "Flight Information"
# 2. Sheduled Time information in UTC and Local and the
time_info_cols = ["CRSDepDateTime","CRSArrDateTime","CRSDepDateTime_UTC", "CRSArrDateTime_UTC","knownWeatherDateTime_UTC",
                  'Year', 'Month','DayofMonth','DayOfWeek',"DayOfYear"]
df_columns.loc[df_columns['column_name'].isin(time_info_cols), 'Category'] = "Time Info"

# 3. Weather Data
weather_infor_cols = ["temp_ARR", "prcp_ARR", "wspd_ARR","temp_DEP", "prcp_DEP", "wspd_DEP"]
df_columns.loc[df_columns['column_name'].isin(weather_infor_cols), 'Category'] = "Weather_info"

# 4. real time statisitcal data collumns
real_time_cols = ["prev_AvgArrDelay","avg_delay"] # there are more, list is incomplete
df_columns.loc[df_columns['column_name'].isin(real_time_cols), 'Category'] = "Real time info"


# put the first and last value of the Collumn into another field, tohave examples

# drop the columns from the dataset
dataset = dataset.drop(columns=all_to_drop)

In [ ]:

# save the collumn info df
display(df_columns)
storage.write_csv(df_columns, destination + "collumns.csv")


In [ ]:
# print all columns with nan values out
print("Columns with nan values:")
for col in dataset.columns:
  print(col)
  if dataset[col].isna().sum() > 0:
    print(f"colum: {col} has nan: {dataset[col].isna().sum()}")

In [ ]:
# dataset.to_feather(destination + output_file_name)
storage.write_parquet(dataset, False, destination + output_file_name)

In [ ]:
print("Saved the dataset to ")
print(destination + output_file_name)
